In [129]:
import json

# Static file with all NDC codes
# Retrieved from https://open.fda.gov/data/downloads/ on April 5, 2025
JSON_DATA = json.load(open('../data/ndc/ndc-codes.json', 'r'))

In [143]:
ARBs = {}
ACEs = {}

def get_NDC_from_pharm_class(pharm_class: str, final_mapping: dict, mode: str = 'file') -> None:
    def add_results(data):
        for drug in data['results']:
            epc_present = drug['openfda'].get('pharm_class_epc') and pharm_class in drug['openfda'].get('pharm_class_epc')
            pharm_class_present = drug.get('pharm_class') and pharm_class in drug.get('pharm_class')
            if epc_present or pharm_class_present:
                generic_name = drug['generic_name'].lower()
                ndcs = [d['package_ndc'] for d in drug['packaging']]
                if generic_name in final_mapping.keys():
                    final_mapping[generic_name].extend(ndcs)
                else:
                    final_mapping[generic_name] = ndcs
                    
    if mode == 'file':
        add_results(JSON_DATA)
    elif mode == 'api':
        response = req.get(f'https://api.fda.gov/drug/ndc.json?search=openfda.pharm_class_epc:"{epc}"&limit=100')
        add_results(resp.json())
        searching = True
        while True:
            Link = urlparse(response.headers.get('Link', '').strip('<').strip('>').split(';')[0]).geturl()
            if not Link:
                break
            response = req.get(Link)
            add_results(resp.json())


In [144]:
get_NDC_from_pharm_class("Angiotensin 2 Receptor Blocker [EPC]", ARBs)
get_NDC_from_pharm_class("Angiotensin Converting Enzyme Inhibitor [EPC]", ACEs)

In [178]:
with open("ARB.json", "w") as f:
    json.dump(ARBs, f)

with open("ACE.json", "w") as f:
    json.dump(ACEs, f)